# Exercise 4: Cross-language comparison

*Morphological Structure · version 0.1.0*

> Segment the Kazakh word *кетпегендіктен* 'because (someone) did not leave' as *кет-пе-ген-дік-тен* (leave-NEG-PTCP-NMLZ-ABL): it contains both a participle *-ген* and a nominaliser *-дік*. (a) Give an equivalent Turkish construction and compare the order of the pieces. (b) Propose a UD annotation under the conventions of the Kazakh KTB treebank, and explain which pieces the UD bundle does not show. Compare it with what the Apertium Kazakh analyser returns for the word. (c) Compare Turkmen *gitmedigi üçin* 'because (someone) did not go' (*git-me-dig-i üçin*, go-NEG-PTCP-POSS.3 for), which expresses the cause with a postposition instead of the ablative.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. Where the compiled `hfst` package has no wheel for your Python version, install `turkicnlp==0.5.0 pyhfst` instead: the first code cell then uses the pure-Python `pyhfst` reader, which has the same lookup interface. The Apertium transducers are downloaded once with `turkicnlp.download`. With `pyhfst`, readings may be listed in a different order than with `hfst`, and the pipeline may select a different reading when several have the same weight.

The code asks the three Apertium analysers how they read the forms and lists how the Kazakh KTB treebank annotates words with the same *-ГАн-дІк* sequence.

In [ ]:
import sys
try:
    import hfst                                   # compiled HFST, as in the chapter
except ImportError:                               # fall back to the pure-Python reader
    import pyhfst
    sys.modules["hfst"] = pyhfst

import turkicnlp
from turkicnlp import Pipeline

def pipeline(lang):
    turkicnlp.download(lang, processors=["morph"])          # Apertium transducers (once)
    return Pipeline(lang, processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")

def analyser(pipe):
    """The Apertium processor of a pipeline (private API in 0.5.0)."""
    return next(p for p in pipe._processors if hasattr(p, "_analyze"))

def readings(pipe, word, plus=False):
    """All transducer readings of one word; by default without '+' continuations (copula etc.)."""
    out = [r["raw"] for r in analyser(pipe)._analyze(word)]
    return out if plus else [r for r in out if "+" not in r]

def show(pipe, sentence):
    for w in pipe(sentence).words:
        print(f"  {w.text:<18} lemma={w.lemma or '_':<10} upos={w.upos or '_':<6} feats={w.feats or '_'}")

tur = pipeline("tur")

In [ ]:
import requests

kaz = pipeline("kaz")
print("Apertium Kazakh:", readings(kaz, "кетпегендіктен"))
tuk = pipeline("tuk")
print("Apertium Turkmen:", [readings(tuk, w) for w in ["gitmedigi", "üçin"]])
print("Apertium Turkish:", readings(tur, "gitmediğinden"), readings(tur, "gitmediği"))

# How the Kazakh KTB treebank annotates -GAndIk forms (UD GitHub, master branch)
BASE = "https://raw.githubusercontent.com/UniversalDependencies/UD_Kazakh-KTB/master/"
for f in ["kk_ktb-ud-train.conllu", "kk_ktb-ud-test.conllu"]:
    sent = None
    for line in requests.get(BASE + f, timeout=60).text.splitlines():
        if line.startswith("# sent_id"):
            sent = line.split("=", 1)[1].strip()
        cols = line.split("\t")
        if len(cols) == 10 and any(x in cols[1].lower() for x in ("гендік", "кендік", "ғандық", "қандық")):
            print(f"KTB {sent}: {cols[1]} lemma={cols[2]} upos={cols[3]} feats={cols[5]} deprel={cols[7]}")

## Worked solution

Output of `experiments/ch05_exercises/run_ch05_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
Apertium Kazakh: ['кет<v><iv><neg><ger_perf><abl>', 'кет<v><tv><neg><ger_perf><abl>', 'кет<vaux><neg><ger_perf><abl>']
Apertium Turkmen: [['git<v><iv><neg><ger_past><acc>', 'git<v><iv><neg><ger_past><px3sp>', 'git<v><iv><neg><gpr_past><acc>', 'git<v><iv><neg><gpr_past><px3sp>'], ['üçin<post>']]
Apertium Turkish: ['git<v><iv><neg><ger_past><px2sg><abl>', 'git<v><iv><neg><ger_past><px3sp><abl>', 'git<v><iv><neg><gpr_past><subst><px2sg><abl>', 'git<v><iv><neg><gpr_past><subst><px3sp><abl>'] ['git<v><iv><neg><ger_past><acc>', 'git<v><iv><neg><ger_past><px3sp><nom>', 'git<v><iv><neg><gpr_past><px3sp>', 'git<v><iv><neg><gpr_past><subst><acc>', 'git<v><iv><neg><gpr_past><subst><px3sp><nom>']
KTB Elizabeth-II.tagged.txt:6:174: тұрғызылғандықтан lemma=тұрғыз upos=VERB feats=Case=Abl|VerbForm=Ger|Voice=Pass deprel=advcl
KTB Elizabeth-II.tagged.txt:9:307: болғандықтан lemma=бол upos=AUX feats=Case=Abl|VerbForm=Ger deprel=cop
KTB Жиырма_Бесінші_Сөз.tagged.txt:12:291: білгендік lemma=біл upos=VERB feats=Case=Nom|VerbForm=Ger deprel=csubj
KTB Иран.tagged.txt:76:2167: екендіктерін lemma=е upos=AUX feats=Case=Acc|Number[psor]=Plur|Person[psor]=3|VerbForm=Ger deprel=cop
```

**(a) Turkish.** *gitmediğinden* 'because he/she did not go' = *git-me-diğ-i-n-den* (go-NEG-PTCP-POSS.3-(n)-ABL), or analytically *gitmediği için*. The order stem – negation – nominalised verb – case is the same as in the Kazakh form. Two pieces differ: Turkish *-DIK* does the work of both *-ген* and *-дік* in one suffix, and it carries a possessive for the subject (*-i*), which the Kazakh form here lacks. The Apertium Turkish analyser reads *gitmediğinden* as `git<v><iv><neg><ger_past><px3sp><abl>` (or with `px2sg`, 'because you did not go').

**(b) Kazakh KTB.** All four KTB words with the *-ГАн-дІк* sequence are annotated `VerbForm=Ger` plus case, for example *тұрғызылғандықтан* `Case=Abl|VerbForm=Ger|Voice=Pass` (advcl). Following that convention: lemma *кет*, `UPOS=VERB`, `Case=Abl|Polarity=Neg|VerbForm=Ger`, relation `advcl`. The bundle does not show that *-ген* and *-дік* are two pieces, nor that *-ген* is a participle; the causal meaning comes from the ablative of a verbal noun and is not a feature. The Apertium analyser also treats *-ген-дік* as one unit (`ger_perf`) and returns three readings that differ only in transitivity or auxiliary use: `кет<v><tv>`, `кет<v><iv>` and `кет<vaux>`, all with `<neg><ger_perf><abl>`.

**(c) Turkmen.** Apertium reads *gitmedigi* as `git<v><iv><neg><ger_past><px3sp>` (or as an accusative) and *üçin* as a postposition (`<post>`). Turkmen therefore expresses the cause like Turkish *gitmediği için*: a verbal noun with a possessive followed by a postposition, where Kazakh uses the ablative case. The Kazakh segmentation and the Turkmen forms still need the native-speaker checks recorded in the chapter.

## Your turn — extension

1. Find a Kazakh *-ГАндІктАн* form in a Kazakh text and check whether it has a possessive (*кеткендігінен*). How does the analyser read it?
2. Look up how the Turkish IMST or BOUN treebank annotates *gitmediği için* and compare with the Kazakh annotation.